In [15]:
import polars as pl
import os

# ================= 配置区域 =================
# 1. 修改点：更新数据根目录
DATA_ROOT = r"../QuantData/Ashare" 
CODE = "600941_SH"
TARGET_PRICE = 102.0
# ===========================================

def analyze():
    print(f"🚀 开始分析 {CODE} ...")
    
    # -----------------------------------------------------------
    # 1. 加载数据 (应用新的时间处理逻辑)
    # -----------------------------------------------------------
    
    # Path handling
    daily_path = os.path.join(DATA_ROOT, "stock_day_raw", f"{CODE}.parquet")
    
    # 2. 修改点：日线行情加载逻辑更新
    # 逻辑：读取 -> 处理时区 -> 剔除头部空值 -> 排序
    df_daily = pl.read_parquet(daily_path)
    df_daily = (
        df_daily
        .with_columns(
            pl.from_epoch("time", time_unit="ms")
            .dt.replace_time_zone("UTC")               # 标记为 UTC
            .dt.convert_time_zone("Asia/Shanghai")     # 转成上海时间
            .dt.date().alias("date")                   # 转为 Date
        )
        .select(["date", "close"])
        .filter(pl.col("close").is_not_null())         # 3. 修改点：只保留有数据的行
        .sort("date")
    )

    # 检查数据起始点
    if df_daily.height > 0:
        start_date = df_daily["date"][0]
        print(f"📅 有效数据起始日期: {start_date}")
    else:
        print("❌ 错误：日线数据全为空！")
        return

    # 股本
    df_cap = pl.read_parquet(os.path.join(DATA_ROOT, "finance_capital", f"{CODE}.parquet"))
    df_cap = df_cap.with_columns(
        pl.col("m_anntime").str.strptime(pl.Date, "%Y%m%d").alias("date"),
        pl.col("total_capital").cast(pl.Float64)
    ).select(["date", "total_capital"]).sort("date")

    # 资产负债
    df_bal = pl.read_parquet(os.path.join(DATA_ROOT, "finance_balance", f"{CODE}.parquet"))
    df_bal = df_bal.with_columns(
        pl.col("m_anntime").str.strptime(pl.Date, "%Y%m%d").alias("date"),
        pl.col("tot_shrhldr_eqy_excl_min_int").cast(pl.Float64).alias("net_assets")
    ).select(["date", "net_assets"]).sort("date")

    # 利润 (用于 TTM 计算)
    df_inc = pl.read_parquet(os.path.join(DATA_ROOT, "finance_income", f"{CODE}.parquet"))
    df_inc = df_inc.with_columns([
        pl.col("m_anntime").str.strptime(pl.Date, "%Y%m%d").alias("pub_date"),
        pl.col("m_timetag").str.strptime(pl.Date, "%Y%m%d").alias("report_date"),
        pl.col("net_profit_excl_min_int_inc").cast(pl.Float64).alias("cum_profit")
    ]).sort("report_date")
    
    # 简单的利润 TTM 计算 (Join 最近一期累计利润 x4 近似，防止 rolling 报错)
    # 注意：这里我们用最简单的逻辑：取 daily 日期对应的最近一期财报
    # 为了防止 Polars 复杂操作报错，我们放到后面 merge 之后做

    # 分红
    df_div = pl.read_parquet(os.path.join(DATA_ROOT, "finance_dividend", f"{CODE}.parquet"))
    df_div = df_div.with_columns(
        pl.col("date").str.strptime(pl.Date, "%Y%m%d"),
        pl.col("interest").cast(pl.Float64)
    ).group_by("date").agg(pl.col("interest").sum()).sort("date")

    # -----------------------------------------------------------
    # 2. 计算分红 TTM (稳健的物理时间轴法)
    # -----------------------------------------------------------
    print("➗ 计算分红 TTM...")
    
    # A. 拿日线的时间范围 (基于清洗后的 df_daily，所以不会从2015年空值开始了)
    min_date = df_daily["date"].min()
    max_date = df_daily["date"].max()
    
    # B. 创建连续日历
    df_calendar = pl.date_range(min_date, max_date, "1d", eager=True).to_frame("date")
    
    # C. 贴上分红数据
    df_calc = df_calendar.join(df_div, on="date", how="left").with_columns(
        pl.col("interest").fill_null(0.0)
    )
    
    # D. 滚动求和 (365天)
    df_calc = df_calc.with_columns(
        pl.col("interest").rolling_sum(window_size=365).alias("div_ttm")
    )
    
    # -----------------------------------------------------------
    # 3. 合并数据
    # -----------------------------------------------------------
    print("🔄 合并所有指标...")
    
    # 以清洗后的 df_daily 为主键
    df_main = df_daily.join_asof(df_cap, on="date", strategy="backward")
    df_main = df_main.join_asof(df_bal, on="date", strategy="backward")
    
    # 利润处理：先生成一个简化版的财报表，再 merge
    df_inc_simple = df_inc.group_by("pub_date").last().rename({"pub_date": "date"}).sort("date")
    df_main = df_main.join_asof(df_inc_simple.select(["date", "cum_profit"]), on="date", strategy="backward")
    
    # 简单估算 TTM 利润：最近一期累计利润 * (4 / 季度数)
    # 季度判断逻辑：根据月份判断是 Q1/Q2/Q3/Q4
    df_main = df_main.with_columns(
        pl.col("date").dt.month().alias("curr_month")
    ).with_columns(
        pl.when(pl.col("curr_month") <= 3).then(4.0) # 假设拿的是年报或Q1? 这里做个简单近似，防止复杂逻辑报错
          .when(pl.col("curr_month") <= 6).then(2.0) # 中报 x 2
          .when(pl.col("curr_month") <= 9).then(1.33)
          .otherwise(1.0)
          .alias("annual_factor")
    ).with_columns(
        (pl.col("cum_profit") * pl.col("annual_factor")).alias("net_profit_ttm_approx")
    )
    
    # 合并分红 TTM
    df_main = df_main.join(df_calc.select(["date", "div_ttm"]), on="date", how="left")

    # -----------------------------------------------------------
    # 4. 最终估值计算
    # -----------------------------------------------------------
    print("🧮 计算估值比率...")
    
    df_final = df_main.with_columns([
        (pl.col("close") * pl.col("total_capital")).alias("market_cap"),
    ]).with_columns([
        (pl.col("market_cap") / pl.col("net_assets")).alias("PB"),
        (pl.col("market_cap") / pl.col("net_profit_ttm_approx")).alias("PE"),
        # 注意：这里的 div_ttm 已经是每股分红的滚动和了，直接除以股价即可
        (pl.col("div_ttm") / pl.col("close") * 100).alias("Yield_Pct")
    ])

    # -----------------------------------------------------------
    # 5. 输出报告
    # -----------------------------------------------------------
    # 过滤掉计算初期的空值
    df_final = df_final.drop_nulls(subset=["PB", "Yield_Pct"])

    if df_final.height == 0:
        print("❌ 结果为空！")
        return

    latest = df_final.tail(1)
    
    curr_price = latest["close"][0]
    curr_date = latest["date"][0]
    curr_yield = latest["Yield_Pct"][0]
    curr_pb = latest["PB"][0]
    
    # 基础数据
    base_div = latest["div_ttm"][0]     # 每股分红 TTM
    base_assets = latest["net_assets"][0] # 归母净资产 (总额)
    base_shares = latest["total_capital"][0] # 总股本

    print(f"\n======== 📊 中国移动 ({CODE}) 估值报告 ========")
    print(f"📅 日期: {curr_date}")
    print(f"💰 现价: {curr_price:.2f}")
    print(f"----------------------------------------")
    print(f"📈 现价股息率: {curr_yield:.2f}%")
    print(f"📘 现价 PB:    {curr_pb:.2f}")
    print(f"----------------------------------------")
    
    print(f"\n🎯 目标价 {TARGET_PRICE} 元 推演:")
    
    # 推演计算
    # 股息率 = 每股分红 / 目标价
    hyp_yield = (base_div / TARGET_PRICE) * 100
    # PB = (目标价 * 总股本) / 归母净资产
    hyp_pb = (TARGET_PRICE * base_shares) / base_assets
    
    print(f"📈 推演股息率: {hyp_yield:.2f}%")
    print(f"📘 推演 PB:    {hyp_pb:.2f}")
    print(f"----------------------------------------")
    
    if hyp_yield > 4.5:
        print("✅ 结论：股息率 > 4.5%，具备底仓配置价值。")
    else:
        print("⚠️ 结论：股息率一般，建议等待更好价格。")

if __name__ == "__main__":
    analyze()

🚀 开始分析 600941_SH ...
📅 有效数据起始日期: 2022-01-05
➗ 计算分红 TTM...
🔄 合并所有指标...
🧮 计算估值比率...

======== 📊 中国移动 (600941_SH) 估值报告 ========
📅 日期: 2025-12-16
💰 现价: 102.99
----------------------------------------
📈 现价股息率: 4.65%
📘 现价 PB:    1.63
----------------------------------------

🎯 目标价 102.0 元 推演:
📈 推演股息率: 4.70%
📘 推演 PB:    1.62
----------------------------------------
✅ 结论：股息率 > 4.5%，具备底仓配置价值。


In [1]:
import polars as pl
import os
import datetime

# ================= 配置区域 =================
DATA_ROOT = r"../QuantData/Ashare" 
CODE = "600941_SH"
# ===========================================

def export_verification_data():
    print(f"🚀 开始生成对账单: {CODE} ...")
    
    # -----------------------------------------------------------
    # 1. 加载并清洗数据
    # -----------------------------------------------------------
    # 日线
    daily_path = os.path.join(DATA_ROOT, "stock_day_raw", f"{CODE}.parquet")
    df_daily = pl.read_parquet(daily_path)
    df_daily = (
        df_daily
        .with_columns(
            pl.from_epoch("time", time_unit="ms")
            .dt.replace_time_zone("UTC")
            .dt.convert_time_zone("Asia/Shanghai")
            .dt.date().alias("date")
        )
        .select(["date", "close"])
        .filter(pl.col("close").is_not_null())
        .sort("date")
    )

    # 股本
    df_cap = pl.read_parquet(os.path.join(DATA_ROOT, "finance_capital", f"{CODE}.parquet"))
    df_cap = df_cap.with_columns(
        pl.col("m_anntime").str.strptime(pl.Date, "%Y%m%d").alias("date"),
        pl.col("total_capital").cast(pl.Float64)
    ).select(["date", "total_capital"]).sort("date")

    # 净资产 (PB分母)
    df_bal = pl.read_parquet(os.path.join(DATA_ROOT, "finance_balance", f"{CODE}.parquet"))
    df_bal = df_bal.with_columns(
        pl.col("m_anntime").str.strptime(pl.Date, "%Y%m%d").alias("date"),
        pl.col("tot_shrhldr_eqy_excl_min_int").cast(pl.Float64).alias("net_assets")
    ).select(["date", "net_assets"]).sort("date")

    # 净利润 (PE分母) - 原始数据
    df_inc = pl.read_parquet(os.path.join(DATA_ROOT, "finance_income", f"{CODE}.parquet"))
    df_inc = df_inc.with_columns([
        pl.col("m_anntime").str.strptime(pl.Date, "%Y%m%d").alias("pub_date"),
        pl.col("m_timetag").str.strptime(pl.Date, "%Y%m%d").alias("report_date"),
        pl.col("net_profit_excl_min_int_inc").cast(pl.Float64).alias("cum_profit")
    ]).sort("pub_date")

    # 分红
    df_div = pl.read_parquet(os.path.join(DATA_ROOT, "finance_dividend", f"{CODE}.parquet"))
    df_div = df_div.with_columns(
        pl.col("date").str.strptime(pl.Date, "%Y%m%d"),
        pl.col("interest").cast(pl.Float64)
    ).group_by("date").agg(pl.col("interest").sum()).sort("date")

    # -----------------------------------------------------------
    # 2. 计算过程 (分红TTM & 利润TTM)
    # -----------------------------------------------------------
    # A. 滚动分红 (物理时间轴)
    min_date = df_daily["date"].min()
    max_date = df_daily["date"].max()
    df_calendar = pl.date_range(min_date, max_date, "1d", eager=True).to_frame("date")
    
    df_div_ttm = df_calendar.join(df_div, on="date", how="left").with_columns(
        pl.col("interest").fill_null(0.0)
    ).with_columns(
        pl.col("interest").rolling_sum(window_size=365).alias("div_ttm")
    )
    
    # B. 利润 TTM (修正版：根据财报月份计算年化系数)
    # 报告期月份：3月(Q1)->x4, 6月(Q2)->x2, 9月(Q3)->x1.33, 12月(Q4)->x1
    df_inc_fix = df_inc.with_columns(
        pl.col("report_date").dt.month().alias("rpt_month")
    ).with_columns(
        pl.when(pl.col("rpt_month") == 3).then(4.0)
          .when(pl.col("rpt_month") == 6).then(2.0)
          .when(pl.col("rpt_month") == 9).then(1.33333333) # 4/3
          .otherwise(1.0)
          .alias("annual_factor")
    )
    # 取最新的财报数据进行合并
    df_inc_simple = df_inc_fix.group_by("pub_date").last().rename({"pub_date": "date"}).sort("date")

    # -----------------------------------------------------------
    # 3. 合并大表
    # -----------------------------------------------------------
    df_main = df_daily.join_asof(df_cap, on="date", strategy="backward")
    df_main = df_main.join_asof(df_bal, on="date", strategy="backward")
    
    # 合并利润表(含cum_profit和annual_factor)
    df_main = df_main.join_asof(
        df_inc_simple.select(["date", "cum_profit", "annual_factor"]), 
        on="date", 
        strategy="backward"
    )
    
    # 在主表上计算 TTM 利润
    df_main = df_main.with_columns(
        (pl.col("cum_profit") * pl.col("annual_factor")).alias("net_profit_ttm_approx")
    )
    
    # 合并分红 TTM
    df_main = df_main.join(df_div_ttm.select(["date", "div_ttm"]), on="date", how="left")

    # -----------------------------------------------------------
    # 4. 导出 CSV (⚠️ 分步计算)
    # -----------------------------------------------------------
    
    # 第一步：先算出 Total Market Cap (总市值)
    df_final = df_main.with_columns([
        (pl.col("close") * pl.col("total_capital")).alias("market_cap")
    ])
    
    # 第二步：再引用 market_cap 算其他指标
    df_final = df_final.with_columns([
        (pl.col("market_cap") / pl.col("net_assets")).alias("PB"),
        (pl.col("market_cap") / pl.col("net_profit_ttm_approx")).alias("PE"),
        (pl.col("div_ttm") / pl.col("close") * 100).alias("Yield_Pct")
    ]).select([
        "date", "close", 
        "total_capital", "net_assets", "net_profit_ttm_approx", "div_ttm",
        "PB", "PE", "Yield_Pct"
    ]).drop_nulls()

    # 导出文件
    csv_path = "check_valuation.csv"
    df_final.write_csv(csv_path)
    print(f"✅ 对账文件已生成: {csv_path}")

    # -----------------------------------------------------------
    # 5. 动态采样与全指标明细打印 (美化版)
    # -----------------------------------------------------------
    print("\n🔎 历史数据抽样核对 (请打开 F10 对照):")
    
    if df_final.height > 0:
        total_rows = df_final.height
        
        # 动态选取三个时间点：最新、4个月前、8个月前
        idx_now = total_rows - 1
        idx_mid = max(0, total_rows - 80)
        idx_far = max(0, total_rows - 160)
        
        # 去重并排序
        target_indices = sorted(list(set([idx_now, idx_mid, idx_far])))
        
        for i in target_indices:
            row = df_final[i]
            curr_date = row["date"][0]
            d_str = str(curr_date)
            
            # 提取基础数值
            close_price = row['close'][0]
            total_shares = row['total_capital'][0]
            mkt_cap_calc = close_price * total_shares # 现场算一下市值
            net_assets = row['net_assets'][0]
            ttm_profit = row['net_profit_ttm_approx'][0]
            ttm_div = row['div_ttm'][0]
            
            print(f"\n📅 采样日期: {d_str}")
            print(f"   {'='*50}")
            
            # --- 板块 1: 市场行情 ---
            print(f"   [1] 市场行情")
            print(f"   收盘价:      {close_price:>10.2f} 元")
            print(f"   总股本:      {total_shares/1e8:>10.2f} 亿股")
            print(f"   总市值:      {mkt_cap_calc/1e8:>10.2f} 亿元  <-- (PB/PE的分子)")
            
            # --- 板块 2: 财务基数 ---
            print(f"\n   [2] 财务基数 (分母)")
            print(f"   归母净资产:  {net_assets/1e8:>10.2f} 亿元  <-- (总市值 / 净资产 = PB)")
            print(f"   TTM净利润:   {ttm_profit/1e8:>10.2f} 亿元  <-- (总市值 / 净利润 = PE)")
            
            # --- 板块 3: 估值指标 ---
            print(f"\n   [3] 估值结果 (代码计算 vs 软件F10)")
            print(f"   PB (市净率): {row['PB'][0]:>10.2f} 倍")
            print(f"   PE (市盈率): {row['PE'][0]:>10.2f} 倍")
            print(f"   股息率(TTM): {row['Yield_Pct'][0]:>10.2f} %")
            
            # --- 板块 4: 分红溯源 ---
            print(f"\n   [4] 股息率溯源")
            print(f"   TTM每股分红: {ttm_div:>10.2f} 元    <-- (分红 / 股价 = 股息率)")
            print(f"   (由以下过去365天内的分红累加):")
            
            # 回溯查找具体分红
            start_date = curr_date - datetime.timedelta(days=365)
            related_divs = df_div.filter(
                (pl.col("date") > start_date) & 
                (pl.col("date") <= curr_date)
            ).sort("date")
            
            if related_divs.height > 0:
                for div_row in related_divs.iter_rows(named=True):
                    div_date = str(div_row['date'])
                    div_amt = div_row['interest']
                    print(f"    ├─ {div_date} 除权: {div_amt:.4f} 元")
            else:
                print(f"    └─ (该时间段内无分红记录)")
                
            print(f"   {'='*50}")

            # --- 板块 5: 前瞻预测 (User's Logic) ---
            print(f"\n   [5] 前瞻预测 (基于2025全年业绩推演)")
            
            # 1. 确定最新的累计利润 (通常是三季报)
            # 我们在前面算 PE 时已经有了 ttm_profit，但那个是滚动过去4个季度的
            # 这里我们需要 "今年前三季度" 的具体数值
            # 逻辑：找离 curr_date 最近的一份财报
            
            # 在 df_inc 中找到 <= curr_date 的最后一行
            latest_report = df_inc.filter(pl.col("pub_date") <= curr_date).tail(1)
            
            if latest_report.height > 0:
                rpt_profit = latest_report["cum_profit"][0]
                rpt_date_val = latest_report["report_date"][0]
                rpt_month = rpt_date_val.month
                
                # 2. 预估 2025 全年利润
                # 简单的线性外推：前三季度(9个月) -> 全年(12个月)
                if rpt_month == 9:
                    est_full_year_profit = rpt_profit / 3 * 4  # 简单粗暴 / 0.75
                    # 或者用之前的 seasonality factor 修正 (移动Q4通常略亏或持平，保守点可以用 /0.78)
                    # 这里为了演示逻辑，用 /0.75 (相当于 annual_factor = 1.33)
                elif rpt_month == 6:
                    est_full_year_profit = rpt_profit * 2
                elif rpt_month == 12:
                    est_full_year_profit = rpt_profit # 已经是全年了
                else:
                    est_full_year_profit = ttm_profit # 兜底
                
                # 3. 设定分红率假设 (移动承诺 >70%，我们按 72% 测算)
                payout_ratio_est = 0.72 
                est_total_div_amt = est_full_year_profit * payout_ratio_est
                
                # 4. 计算每股分红 (EPS Div)
                est_eps_div = est_total_div_amt / total_shares
                
                # 5. 计算已知的中期分红 (Helper)
                # 我们需要找今年发过的哪笔是“中期”的。通常 9月份发的那笔是中期。
                # 逻辑：找最近半年内的一笔分红
                mid_div_search = df_div.filter(
                    (pl.col("date") > (curr_date - datetime.timedelta(days=180))) &
                    (pl.col("date") <= curr_date)
                )
                known_interim_div = 0.0
                if mid_div_search.height > 0:
                    known_interim_div = mid_div_search["interest"].sum()
                
                # 6. 倒挤出“明年的大红包” (末期股息)
                est_final_div = max(0, est_eps_div - known_interim_div)
                
                # 7. 计算前瞻股息率
                forward_yield = (est_eps_div / close_price) * 100
                
                print(f"   当前财报:    {rpt_date_val} (Q{rpt_month//3})")
                print(f"   已获净利润:  {rpt_profit/1e8:.2f} 亿元")
                print(f"   预估全利润:  {est_full_year_profit/1e8:.2f} 亿元 (线性外推)")
                print(f"   分红率假设:  {payout_ratio_est*100:.0f}% (保守估计)")
                print(f"   ----------------------------------------")
                print(f"   预估全年分红: {est_eps_div:.2f} 元/股")
                print(f"   - 已发中期:   {known_interim_div:.2f} 元/股 (假设已发)")
                print(f"   = 待发末期:   {est_final_div:.2f} 元/股 (预计明年6月发)")
                print(f"   ----------------------------------------")
                print(f"   ★ 前瞻股息率: {forward_yield:.2f}%")
                
                if forward_yield > row['Yield_Pct'][0]:
                    print(f"   📈 结论: 前瞻 > TTM，业绩在增长，现在买更划算！")
                else:
                    print(f"   📉 结论: 前瞻 < TTM，业绩可能下滑或分红不及预期。")
            else:
                print("   (无财报数据，无法预测)")

            print(f"   {'='*50}")
            # --- 板块 6: 2026年 收益敏感性分析 (Breaking the Limit) ---
            print(f"\n   [6] 极限推演：2026年收益敏感性矩阵")
            print(f"   (假设持有到2026年9月，验证策略含金量)")
            print(f"   基准参数: 2025预估利润 = {est_full_year_profit/1e8:.2f} 亿")
            
            # 定义情景变量
            # 1. 净利润增长率假设 (2026 vs 2025)
            growth_scenarios = [-0.02, 0.00, 0.03, 0.05, 0.08] # -2% 到 +8%
            # 2. 分红率假设 (移动承诺是逐步提升)
            payout_scenarios = [0.70, 0.72, 0.74, 0.75] 
            
            print(f"   {'-'*65}")
            print(f"   {'利润增长':<10} | {'分红率 70%':<10} | {'分红率 72%':<10} | {'分红率 74%':<10} | {'分红率 75%':<10}")
            print(f"   {'-'*65}")
            
            # 循环计算矩阵
            for g in growth_scenarios:
                row_str = f"   {g*100:>+6.0f}%    |"
                
                for p in payout_scenarios:
                    # 2026 预估利润
                    profit_2026 = est_full_year_profit * (1 + g)
                    # 2026 预估分红 (全口径)
                    div_2026 = (profit_2026 * p) / total_shares
                    # 对应当前股价的股息率
                    yield_2026 = (div_2026 / close_price) * 100
                    
                    # 格式化输出 (高亮 > 6% 的情况, 或者是 > 5%)
                    val_str = f"{yield_2026:.2f}%"
                    if yield_2026 >= 6.0:
                        val_str += "🔥" # 极佳
                    elif yield_2026 >= 5.0:
                        val_str += "✅" # 达标
                    else:
                        val_str += "  "
                        
                    row_str += f" {val_str:<10} |"
                
                print(row_str)
            
            print(f"   {'-'*65}")
            print(f"   注：此表展示的是当你持有到2026年完整周期时，基于当前成本(102元)的潜在回报率。")
            print(f"   {'='*50}")

            # --- 板块 7: 港股 (0941.HK) 比价透视 ---
            print(f"\n   [7] 港股 (0941.HK) 跨市场比价")
            
            # 1. 设定参数 (需要你手动输入当天的港股价格和汇率)
            # 假设数据：2025-12-16 的大概数值
            price_hkd = 83.70  # 港股现价 (假设值，需查软件)
            exchange_rate = 0.9051 # 1 HKD = 0.92 RMB (假设汇率)
            
            # 2. 计算 RMB 等值价格
            price_h_rmb = price_hkd * exchange_rate
            
            # 3. 计算 AH 溢价率
            # 公式：(A股价格 - H股RMB价格) / H股RMB价格
            ah_premium = (close_price - price_h_rmb) / price_h_rmb * 100
            
            # 4. 计算 H 股股息率
            # 逻辑：分红金额(RMB)是一样的，除以更便宜的 H股(RMB)价格
            yield_h_ttm = (ttm_div / price_h_rmb) * 100
            yield_h_fwd = (5.12 / price_h_rmb) * 100 # 使用之前算出的前瞻分红 5.12元
            
            print(f"   港股现价:    {price_hkd:.2f} HKD  (≈ {price_h_rmb:.2f} RMB)")
            print(f"   A股现价:     {close_price:.2f} RMB")
            print(f"   ----------------------------------------")
            print(f"   📉 H股折价:  A股比H股贵 {ah_premium:.1f}%")
            print(f"   💰 H股前瞻股息率: {yield_h_fwd:.2f}% (对比A股 {forward_yield:.2f}%)")
            
            # 5. 港股通红利税警告 (必看!)
            print(f"   ----------------------------------------")
            print(f"   ⚠️ 真实到手收益率警告 (港股通用户):")
            print(f"   港股通扣税:  20% (红利税)")
            print(f"   A股扣税:     0% (持有>1年免税)")
            
            real_yield_h = yield_h_fwd * 0.8
            real_yield_a = forward_yield * 1.0
            
            print(f"   H股税后:     {real_yield_h:.2f}%")
            print(f"   A股税后:     {real_yield_a:.2f}%")
            
            if real_yield_h > real_yield_a:
                print(f"   ✅ 结论: 即使扣完20%的税，港股依然更香！")
            else:
                print(f"   ❌ 结论: 港股看着便宜，扣完税不如直接买A股！")
                
            print(f"   {'='*50}")

    else:
        print("❌ 结果集为空，无法打印采样。")

if __name__ == "__main__":
    export_verification_data()

🚀 开始生成对账单: 600941_SH ...
✅ 对账文件已生成: check_valuation.csv

🔎 历史数据抽样核对 (请打开 F10 对照):

📅 采样日期: 2025-04-23
   [1] 市场行情
   收盘价:          110.84 元
   总股本:          215.84 亿股
   总市值:        23924.01 亿元  <-- (PB/PE的分子)

   [2] 财务基数 (分母)
   归母净资产:    13903.16 亿元  <-- (总市值 / 净资产 = PB)
   TTM净利润:      1225.24 亿元  <-- (总市值 / 净利润 = PE)

   [3] 估值结果 (代码计算 vs 软件F10)
   PB (市净率):       1.72 倍
   PE (市盈率):      19.53 倍
   股息率(TTM):       4.12 %

   [4] 股息率溯源
   TTM每股分红:       4.56 元    <-- (分红 / 股价 = 股息率)
   (由以下过去365天内的分红累加):
    ├─ 2024-06-06 除权: 2.1849 元
    ├─ 2024-09-02 除权: 2.3789 元

   [5] 前瞻预测 (基于2025全年业绩推演)
   当前财报:    2025-03-31 (Q1)
   已获净利润:  306.31 亿元
   预估全利润:  1225.24 亿元 (线性外推)
   分红率假设:  72% (保守估计)
   ----------------------------------------
   预估全年分红: 4.09 元/股
   - 已发中期:   0.00 元/股 (假设已发)
   = 待发末期:   4.09 元/股 (预计明年6月发)
   ----------------------------------------
   ★ 前瞻股息率: 3.69%
   📉 结论: 前瞻 < TTM，业绩可能下滑或分红不及预期。

   [6] 极限推演：2026年收益敏感性矩阵
   (假设持有到2026年9月，验证策略含金量)
   基准参数: 2025预估利润 = 122